# Task 4: robust training

Three methods for learning with noisy labels, each under the same budget as the Task 1 noisy baseline (60 epochs per network, at most two networks) and with the same optimizer, schedule and augmentation:

1. **relabel**: uses the Task 3 out-of-fold predictions. A training image keeps its noisy label when the cross-validated models agree with it; it is relabelled to the cross-validated prediction when they disagree confidently (averaged probability ≥ 0.5); otherwise it is dropped. A fresh ResNet-18 is then trained with cross-entropy on the result.
2. **sce**: symmetric cross-entropy (α = 0.1, β = 1.0, the CIFAR-10 setting of Wang et al., 2019). The reverse term gives bounded loss on samples the network confidently disagrees with, so wrong labels pull less.
3. **coteaching**: two networks trained together (Han et al., 2018). Each keeps the small-loss part of every batch and teaches the other network with it. The forget rate grows to the noise rate estimated in Task 3 from CV disagreement (no clean labels) over the first 10 epochs. Predictions average the two networks.

For every method the checkpoint epoch and the best method are chosen by noisy validation accuracy. Clean validation accuracy is reported next to it as a measurement only.

Runtime on an A100: about 20 minutes (co-teaching trains two networks).

In [1]:
# project folder: MyDrive/cv_bootcamp (data/, model.py, check_format.py, results_template.json, code/cvlib.py).
# If it is not there yet, it is unpacked from MyDrive/cv_bootcamp.zip, or from a zip picked in the file dialog.
import sys, zipfile
from pathlib import Path

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/cv_bootcamp")
except ImportError:                      # running outside Colab
    files = None
    ROOT = Path.cwd() / "cv_bootcamp"

if not (ROOT / "data" / "labels.csv").exists():
    zips = [z for z in [ROOT.parent / "cv_bootcamp.zip"] if z.exists()] + list(Path.cwd().glob("cv_bootcamp*.zip"))
    if not zips:
        zips = [Path(name) for name in files.upload()]
    zipfile.ZipFile(zips[0]).extractall(ROOT.parent)
    print("unpacked", zips[0].name, "into", ROOT.parent)

assert (ROOT / "data" / "labels.csv").exists() and (ROOT / "code" / "cvlib.py").exists()
sys.path.insert(0, str(ROOT / "code"))
print("project folder:", ROOT)

Mounted at /content/drive
project folder: /content/drive/MyDrive/cv_bootcamp


In [2]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cvlib as cv

WORK = ROOT / "work"
PRED = ROOT / "predictions"
WORK.mkdir(exist_ok=True)
PRED.mkdir(exist_ok=True)

print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
data = cv.Data(ROOT)
print("train", len(data.train_idx), " val", len(data.val_idx), " test", len(data.x_test))
print("classes:", data.classes)

device: NVIDIA A100-SXM4-80GB
train 45000  val 5000  test 10000
classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
lab = data.labels
tr, va = data.train_idx, data.val_idx
noisy, clean = lab["noisy_label"].to_numpy(), lab["clean_label"].to_numpy()
y_tr = noisy[tr]

oof = np.load(WORK / "t3_oof.npz")
oof_avg, est_noise = oof["oof_avg"], float(oof["est_noise_rate"])
print(f"noise rate estimated in Task 3 (no clean labels): {est_noise:.4f}")

METHODS = ["relabel", "sce", "coteaching"]
RELABEL_CONF = 0.5

noise rate estimated in Task 3 (no clean labels): 0.4398


## Relabelled training set

In [4]:
pos = np.arange(len(tr))
cv_pred = oof_avg.argmax(1)
cv_conf = oof_avg.max(1)

agree = cv_pred == y_tr
relabel = ~agree & (cv_conf >= RELABEL_CONF)
keep = agree | relabel
new_labels = np.where(relabel, cv_pred, y_tr)

print(f"kept as given: {agree.sum()}   relabelled: {relabel.sum()}   dropped: {(~keep).sum()}   training on: {keep.sum()}")
# measurement only: how clean the new training set is
wrong_before = (y_tr != clean[tr]).mean()
wrong_after = (new_labels[keep] != clean[tr][keep]).mean()
print(f"label noise before {wrong_before:.4f}, after {wrong_after:.4f} (clean labels used only for this print)")
cv.update_results(WORK, extra={"relabel_counts": {"kept": int(agree.sum()), "relabelled": int(relabel.sum()), "dropped": int((~keep).sum())},
                               "relabel_noise_after": round(float(wrong_after), 4)})

kept as given: 25208   relabelled: 6470   dropped: 13322   training on: 31678
label noise before 0.4024, after 0.0695 (clean labels used only for this print)


{'task1': {'noise_rate': 0.4024,
  'val_acc_noisy_labels__clean_ref_model': 0.5918,
  'val_acc_noisy_labels__noisy_ce_model': 0.4924},
 'seed': 42,
 'training_setup': {'optimizer': 'SGD (momentum 0.9, nesterov, weight decay 5e-4)',
  'learning_rate_schedule': 'cosine annealing per step, 0.1 to 0 over 60 epochs, no warmup',
  'batch_size': 128,
  'epochs': 60,
  'augmentation': 'random crop 32x32 with 4-pixel zero padding, random horizontal flip'},
 'extra': {'clean_ref_val_acc_clean': 0.9446,
  'noisy_ce_val_acc_clean_final': 0.6844,
  'noisy_ce_val_acc_clean_early_stop': 0.779,
  'clean_val_peak_epoch': 43,
  't3_auroc': 0.9418,
  't3_precision_at_k': 0.8307,
  't3_check': {'cv_confidence (submitted)': {'AUROC': 0.9418,
    'precision@k': 0.8307},
   'cv_last_epoch': {'AUROC': 0.8774, 'precision@k': 0.7637},
   'early_loss': {'AUROC': 0.9371, 'precision@k': 0.8172},
   'cv_disagreement': {'AUROC': 0.8834, 'precision@k': 0.8222},
   'combined_rank': {'AUROC': 0.9417, 'precision@k': 0.8

## Training the methods

In [ ]:
def run_method(name):
    path = WORK / f"t4_{name}.npz"
    if path.exists():
        return dict(np.load(path))
    rec = cv.Recorder(data, noisy[va])
    if name == "relabel":
        cv.train(data, tr[keep], new_labels[keep], on_epoch=rec, tag=name)
    elif name == "sce":
        cv.train(data, tr, y_tr, loss_fn=cv.sce_loss, on_epoch=rec, tag=name)
    elif name == "coteaching":
        cv.train_coteaching(data, tr, y_tr, forget_rate=est_noise, on_epoch=rec, tag=name)
    rec.save(path)
    return dict(np.load(path))

runs = {name: run_method(name) for name in METHODS}

relabel epoch  1  loss 2.3839  val_acc 0.2080  (17s)
relabel epoch  5  loss 0.9312  val_acc 0.4284  (29s)
relabel epoch 10  loss 0.4449  val_acc 0.4312  (44s)
relabel epoch 15  loss 0.2711  val_acc 0.5154  (59s)
relabel epoch 20  loss 0.2152  val_acc 0.5358  (74s)
relabel epoch 25  loss 0.1787  val_acc 0.5192  (89s)
relabel epoch 30  loss 0.1420  val_acc 0.5280  (104s)
relabel epoch 35  loss 0.1030  val_acc 0.5364  (119s)
relabel epoch 40  loss 0.0735  val_acc 0.5616  (134s)
relabel epoch 45  loss 0.0351  val_acc 0.5638  (149s)
relabel epoch 50  loss 0.0148  val_acc 0.5738  (164s)
relabel epoch 55  loss 0.0078  val_acc 0.5714  (179s)
relabel epoch 60  loss 0.0068  val_acc 0.5730  (194s)
sce epoch  1  loss 7.6438  val_acc 0.2976  (8s)
sce epoch  5  loss 6.3350  val_acc 0.3816  (26s)
sce epoch 10  loss 5.7029  val_acc 0.3834  (48s)
sce epoch 15  loss 5.3544  val_acc 0.3894  (69s)
sce epoch 20  loss 5.0616  val_acc 0.4656  (91s)
sce epoch 25  loss 4.8920  val_acc 0.4666  (113s)
sce epoch 

In [ ]:
rows, curves = [], {}
for name, r in runs.items():
    vp = r["val_probs"].astype(np.float32)
    acc_n = (vp.argmax(2) == noisy[va]).mean(1)
    acc_c = (vp.argmax(2) == clean[va]).mean(1)
    best = int(acc_n.argmax()) + 1
    cv.write_preds(PRED / f"t4_{name}.csv", r["test_probs"][best - 1].astype(np.float32))
    curves[name] = (acc_n, acc_c)
    rows.append({"method": name, "best epoch": best, "val acc noisy (best)": acc_n[best - 1],
                 "val acc clean (best)": acc_c[best - 1], "val acc noisy (final)": acc_n[-1], "val acc clean (final)": acc_c[-1]})

# reference rows from Tasks 1 and 2
d = np.load(WORK / "t1_noisy_ce.npz")
dyn = pd.DataFrame(d["dynamics"], columns=[str(c) for c in d["dynamics_cols"]])
es = int(dyn.val_acc_noisy.to_numpy().argmax())
rows.append({"method": "CE (Task 1, final)", "best epoch": len(dyn), "val acc noisy (best)": dyn.val_acc_noisy.iloc[-1],
             "val acc clean (best)": dyn.val_acc_clean.iloc[-1], "val acc noisy (final)": dyn.val_acc_noisy.iloc[-1], "val acc clean (final)": dyn.val_acc_clean.iloc[-1]})
rows.append({"method": "CE early stop (Task 2)", "best epoch": es + 1, "val acc noisy (best)": dyn.val_acc_noisy.iloc[es],
             "val acc clean (best)": dyn.val_acc_clean.iloc[es], "val acc noisy (final)": dyn.val_acc_noisy.iloc[-1], "val acc clean (final)": dyn.val_acc_clean.iloc[-1]})
table = pd.DataFrame(rows).set_index("method").round(4)
display(table)

best_method = max(METHODS, key=lambda n: table.loc[n, "val acc noisy (best)"])
print("best method (noisy val acc):", best_method)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (acc_n, acc_c) in curves.items():
    ep = np.arange(1, len(acc_n) + 1)
    ax[0].plot(ep, acc_n, label=name)
    ax[1].plot(ep, acc_c, label=name)
ep = dyn["epoch"].astype(int)
ax[0].plot(ep, dyn.val_acc_noisy, "k--", label="CE")
ax[1].plot(ep, dyn.val_acc_clean, "k--", label="CE")
ax[0].set_title("Validation accuracy, noisy labels (used for selection)")
ax[1].set_title("Validation accuracy, clean labels (measurement only)")
for a in ax:
    a.set_xlabel("epoch"); a.set_ylabel("accuracy"); a.grid(alpha=0.3); a.legend()
fig.tight_layout(); fig.savefig(WORK / "t4_methods.png", dpi=150); plt.show()

cv.update_results(WORK, task4={
    "methods": [{"name": n, "file": f"t4_{n}.csv", "val_acc_noisy_labels": round(float(table.loc[n, "val acc noisy (best)"]), 4)} for n in METHODS],
    "best_method": best_method,
}, extra={"t4_table": table.reset_index().to_dict("records")})
print(json.dumps(cv.load_results(WORK)["task4"], indent=2))